# One look, six answers: a pilot on Kaggle

Apertus v1.5 8B reads each image **once**. Nothing is generated and nothing is fine-tuned: we keep its hidden states
at several depths and fit one small linear classifier per dimension on them. This notebook answers one question
before anyone builds more: **how far do frozen Apertus features get on a sample of train.csv?**

It reads about 1,500 images from the part of train.csv outside the official holdout, each one as it is and mirrored,
and prints one block to paste into the PR: the score per dimension, which depth of Apertus works best, three ways to
handle left and right in perspective, a learning curve, and the speed. No image id is printed.

**Kaggle settings, before "Run all"**

1. Session options: Accelerator **GPU T4 x2**, Internet **on** (needs a phone-verified Kaggle account).
2. Add Input: Competitions, **hackapertus-galaxus**.
3. Add-ons, Secrets: **HF_TOKEN**, a Hugging Face read token of an account that has access to
   [swiss-ai/Apertus-v1.5-8B](https://huggingface.co/swiss-ai/Apertus-v1.5-8B) (request access on that page; it is
   approved automatically). Tick it for this notebook.
4. Keep the notebook **private**. The features in /kaggle/working come from competition data: never publish them,
   never download them into a repository.

"Save Version, Save & Run All" runs it in the background, so the browser can close. The settings cell is the only one
to edit.

In [ ]:
import subprocess
import sys

# transformers 5.24.1 is the first release we know loads Apertus 1.5 (Apertus1p5ForConditionalGeneration).
PACKAGES = ["transformers==5.24.1", "accelerate>=1.2"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PACKAGES], check=True)

In [ ]:
from pathlib import Path

MODEL_ID = "swiss-ai/Apertus-v1.5-8B"
MODEL_REVISION = (
    "1519de0ae7f3d114879291810a58259966f64c99"  # main on 2026-11-27, pinned so a rerun sees the same weights
)
PILOT_IMAGES = 1500  # from the non-holdout part of train.csv; each one is read twice, as it is and mirrored
IMAGE_SIZE = 512  # about 512 x 512 pixels, aspect ratio kept, as the Galaxus Qwen baseline
PATCH = 32  # sides rounded to a multiple of 32, so Apertus' 16-pixel tokenizer never resizes again
DTYPE = "float16"  # a T4 has no bfloat16; on an L40S, A100 or H100 use "bfloat16"
USE_CONTEXT = True  # product title, brand, type and category in the prompt
SEED = 0
FOLDS = 5  # out-of-fold by product: no product is in training and test at once
C_GRID = (0.05, 0.5)  # inverse L2 strength of the logistic heads
PCA_DIMS = 256  # each feature block is reduced to this many components before the heads
WORK = Path("/kaggle/working/one_look")
WORK.mkdir(parents=True, exist_ok=True)

In [ ]:
import importlib.util
import sys

import numpy as np
import pandas as pd

INPUT = Path("/kaggle/input")
train_files = sorted(INPUT.rglob("train.csv"))
metric_files = sorted(INPUT.rglob("metric.py"))
if not train_files or not metric_files:
    raise FileNotFoundError("Add the competition data: Add Input, Competitions, hackapertus-galaxus")
spec = importlib.util.spec_from_file_location("metric", metric_files[0])
metric = importlib.util.module_from_spec(spec)
sys.modules["metric"] = metric
spec.loader.exec_module(metric)

# Dimensions and classes come from the competition's metric.py; this notebook types none of them.
DIMENSIONS = list(metric.DIMENSIONS)
CLASSES = {dimension: sorted(metric.VALID_CLASSES[dimension]) for dimension in DIMENSIONS}
# The perspective dimension is the one whose classes come in left and right pairs: found, not typed.
SIDED = [d for d in DIMENSIONS if any("left" in c for c in CLASSES[d]) and any("right" in c for c in CLASSES[d])]
PERSPECTIVE = SIDED[0] if SIDED else None


def swap_side(label):
    """The class a horizontal mirror turns this one into, left and right swapped; any other class stays."""
    swapped = label.replace("left", "<L>").replace("right", "left").replace("<L>", "right")
    return swapped if PERSPECTIVE is not None and swapped in CLASSES[PERSPECTIVE] else label


def read_part(csv_path):
    """A competition csv as strings, with the full path of each image."""
    frame = pd.read_csv(csv_path, dtype=str, keep_default_na=False)
    frame["image_file"] = [str(csv_path.parent / image_path) for image_path in frame["image_path"]]
    return frame


def scored(frame, dimension):
    """The cells metric.py scores: a human label where the dimension is relevant."""
    relevant = frame[f"{dimension}_relevant"].astype(str).str.strip().str.lower().isin(["true", "1", "yes"])
    return ((frame[f"{dimension}_source"] != "model") & relevant).to_numpy()


train = read_part(train_files[0])
# The official holdout of challenge_description.md "Scoring locally" stays untouched for later EXP rows.
# Copied for Kaggle self-containment: these must stay equal to galaxus.data.split.HOLDOUT_FRAC and HOLDOUT_SEED
# (src/galaxus/data/split.py), where the official holdout is defined. tests/model/test_model_one_look_pilot.py
# checks the copy in ci, and the check below stops the run if the repo is importable and they differ.
HOLDOUT_FRAC, HOLDOUT_SEED = 0.2, 0
try:
    from galaxus.data import split as official_split
except ImportError:  # on Kaggle the repo is not installed; the ci test above guards the copy
    official_split = None
if official_split is not None and (official_split.HOLDOUT_FRAC, official_split.HOLDOUT_SEED) != (HOLDOUT_FRAC, HOLDOUT_SEED):
    raise RuntimeError(
        f"holdout constants {(HOLDOUT_FRAC, HOLDOUT_SEED)} differ from galaxus.data.split "
        f"{(official_split.HOLDOUT_FRAC, official_split.HOLDOUT_SEED)}: the pilot would train on holdout products"
    )
holdout_products = set(train["product_id"].drop_duplicates().sample(frac=HOLDOUT_FRAC, random_state=HOLDOUT_SEED))
rest = train[~train["product_id"].isin(holdout_products)]
images_per_product = rest["product_id"].value_counts()
picked, count = [], 0
for product in rest["product_id"].drop_duplicates().sample(frac=1.0, random_state=SEED):
    if count >= PILOT_IMAGES:
        break
    picked.append(product)
    count += int(images_per_product[product])
pilot = rest[rest["product_id"].isin(set(picked))].reset_index(drop=True)
print(f"pilot: {len(pilot)} images of {pilot['product_id'].nunique()} products (train.csv has {len(train)})")
for dimension in DIMENSIONS:
    counts = pilot.loc[scored(pilot, dimension), dimension].value_counts()
    print(
        f"  {dimension}: {int(counts.sum())} scored cells, {len(counts)} of {len(CLASSES[dimension])} classes seen, "
        f"rarest {int(counts.min()) if len(counts) else 0}"
    )

In [ ]:
import io

from PIL import Image as PillowImage
from PIL import ImageOps


def opaque_rgb_on_white(image):
    """RGB, with any transparency composited onto white (the function on the Kaggle data page)."""
    if image.mode not in ("RGBA", "LA", "PA") and "transparency" not in image.info:
        return image.convert("RGB")
    rgba = image.convert("RGBA")
    canvas = PillowImage.new("RGBA", rgba.size, (255, 255, 255, 255))
    canvas.alpha_composite(rgba)
    return canvas.convert("RGB")


def native_aspect_image(image):
    """About IMAGE_SIZE squared pixels, aspect ratio kept, both sides a multiple of PATCH."""
    scale = IMAGE_SIZE / (image.width * image.height) ** 0.5
    scale *= min(1.0, 8 * IMAGE_SIZE / (max(image.width, image.height) * scale))
    width = max(PATCH, round(image.width * scale / PATCH) * PATCH)
    height = max(PATCH, round(image.height * scale / PATCH) * PATCH)
    return image.resize((width, height), PillowImage.LANCZOS)


def prepared(image_file):
    """The pixels Apertus sees: white background, resized, through JPEG at quality 92 like the Qwen baseline."""
    with PillowImage.open(image_file) as image:
        rgb = opaque_rgb_on_white(image)
    buffer = io.BytesIO()
    native_aspect_image(rgb).save(buffer, format="JPEG", quality=92)
    return PillowImage.open(io.BytesIO(buffer.getvalue())).convert("RGB")

In [ ]:
import os
import time

import torch
import transformers
from transformers import AutoProcessor

print("transformers", transformers.__version__, "| torch", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("No GPU: Session options, Accelerator, GPU T4 x2")
GPUS = [torch.cuda.get_device_name(index) for index in range(torch.cuda.device_count())]
print("gpus:", GPUS)


def hf_token():
    """The HF_TOKEN Kaggle secret, else the environment variable. Never printed."""
    try:
        from kaggle_secrets import UserSecretsClient
    except ImportError:
        return os.environ.get("HF_TOKEN")
    return UserSecretsClient().get_secret("HF_TOKEN")


model_class = getattr(transformers, "Apertus1p5ForConditionalGeneration", None)
if model_class is None:
    raise ImportError(f"transformers {transformers.__version__} has no Apertus 1.5: rerun the install cell, restart")
# Leave room on each card for the vision tokenizer's activations (it runs in float32).
max_memory = {
    index: f"{int(torch.cuda.get_device_properties(index).total_memory * 0.85 / 2**30)}GiB"
    for index in range(torch.cuda.device_count())
}
started = time.perf_counter()
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION, token=hf_token())
model = model_class.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    token=hf_token(),
    dtype=getattr(torch, DTYPE),
    device_map="auto",
    max_memory=max_memory,
)
model.eval()
LOAD_SECONDS = time.perf_counter() - started
IMAGE_TOKEN_ID = processor.image_token_id
if IMAGE_TOKEN_ID != getattr(model.config, "image_token_id", IMAGE_TOKEN_ID):
    raise RuntimeError(
        f"processor and model disagree on the image token: {IMAGE_TOKEN_ID} vs {model.config.image_token_id}"
    )
SPATIAL = processor.image_processor.spatial_factor
print(f"loaded {MODEL_ID} in {LOAD_SECONDS:.0f} s; one image token per {SPATIAL} x {SPATIAL} pixels")

In [ ]:
def decoder_layers(network):
    """The language model's decoder layers: the longest ModuleList of *DecoderLayer modules."""
    candidates = [
        module
        for module in network.modules()
        if isinstance(module, torch.nn.ModuleList) and len(module) and "DecoderLayer" in type(module[0]).__name__
    ]
    if not candidates:
        raise RuntimeError("found no decoder layers in the model")
    return max(candidates, key=len)


LAYERS = decoder_layers(model)
N_LAYERS = len(LAYERS)
# Depth 0 is the embedded tokens entering the first layer (for an image: its discrete visual codes); depth k is the
# output of layer k. The deepest one read is the last layer, so the final norm and the vocabulary head never run.
PROBE = sorted({0, *range(4, N_LAYERS, 4), N_LAYERS})
DEEPEST = max(PROBE)
# last: the last prompt token, where the answer would start. img: the mean over the image tokens.
# lr: the mean over the left half and the mean over the right half of the image grid, side by side.
POOLS = ("last", "img", "lr")
CONTEXT_COLUMNS = (
    ("Product", "product_title"),
    ("Brand", "brand"),
    ("Product type", "product_type"),
    ("Category", "category"),
    ("Subcategory", "subcategory"),
)
QUESTION = (
    "Look at this product image from an online shop. What does it show, what is behind the product, from which side "
    "is the product seen, is a real person or animal in it, which kind of text is printed on it, and in which language?"
)


class EarlyExit(Exception):
    """Raised after the deepest layer we read, so the rest of the model never runs."""


state = {"captured": {}, "where": None}


def pooled(hidden):
    """The three pools of one sequence of hidden states, shape (tokens, hidden size)."""
    image, left, right = (index.to(hidden.device) for index in state["where"])
    hidden = hidden.float()
    return {
        "last": hidden[-1].cpu().numpy().astype(np.float16),
        "img": hidden[image].mean(0).cpu().numpy().astype(np.float16),
        "lr": torch.cat([hidden[left].mean(0), hidden[right].mean(0)]).cpu().numpy().astype(np.float16),
    }


def before_first_layer(module, args, kwargs):
    hidden = args[0] if args else kwargs["hidden_states"]
    state["captured"][0] = pooled(hidden[0])


def after_layer(depth):
    def hook(module, args, output):
        hidden = output[0] if isinstance(output, tuple) else output
        state["captured"][depth] = pooled(hidden[0])
        if depth == DEEPEST:
            raise EarlyExit

    return hook


for handle in globals().get("HOOKS", []):  # running this cell twice must not stack hooks
    handle.remove()
HOOKS = [LAYERS[0].register_forward_pre_hook(before_first_layer, with_kwargs=True)]
HOOKS += [LAYERS[depth - 1].register_forward_hook(after_layer(depth)) for depth in PROBE if depth > 0]


def prompt_text(row):
    """The product context, if used, then the question."""
    context = [f"{name}: {row.get(column, '')}" for name, column in CONTEXT_COLUMNS if USE_CONTEXT and row.get(column)]
    return "\n".join([*context, QUESTION])


def read_once(image, text):
    """One pass of Apertus over the image and the prompt: the pooled hidden states at every PROBE depth."""
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": text}]}]
    chat = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    # The chat template already starts with <s>; letting the tokenizer add another would double it.
    inputs = processor(text=[chat], images=[image], return_tensors="pt", add_special_tokens=False)
    ids = inputs["input_ids"][0]
    image_positions = (ids == IMAGE_TOKEN_ID).nonzero().flatten()
    height, width = (int(side) for side in inputs["image_sizes"][0])
    rows, cols = height // SPATIAL, width // SPATIAL
    if len(image_positions) != rows * cols:
        raise RuntimeError(f"{len(image_positions)} image tokens for a {rows} x {cols} grid")
    column = torch.arange(rows * cols) % cols  # the image tokens come row by row
    half = cols // 2
    state["where"] = (image_positions, image_positions[column < half], image_positions[column >= cols - half])
    state["captured"] = {}
    device = model.get_input_embeddings().weight.device
    batch = {key: value.to(device) if torch.is_tensor(value) else value for key, value in inputs.items()}
    try:
        with torch.inference_mode():
            model(**batch, use_cache=False)
    except EarlyExit:
        pass
    missing = [depth for depth in PROBE if depth not in state["captured"]]
    if missing:
        raise RuntimeError(f"no hidden state at depths {missing}")
    return dict(state["captured"]), int(ids.numel())

In [ ]:
# Smoke test: three images. Stops here, with the reason, if Apertus does not load or overflows in float16.
for _, row in pilot.head(3).iterrows():
    image = prepared(row["image_file"])
    started = time.perf_counter()
    features, TOKENS = read_once(image, prompt_text(row))
    seconds = time.perf_counter() - started
    finite = all(
        np.isfinite(vector.astype(np.float32)).all() for pools in features.values() for vector in pools.values()
    )
    print(f"{image.width} x {image.height} px -> {TOKENS} tokens, {seconds:.2f} s, finite: {finite}")
    if not finite:
        raise FloatingPointError(
            "NaN or inf in the hidden states: float16 overflows on this card. Run on an L40S/A100 with "
            "DTYPE = 'bfloat16' and say so on the PR."
        )
REFERENCE = features[DEEPEST]
print(f"depths {PROBE} of {N_LAYERS} | sizes { ({pool: vector.shape[0] for pool, vector in REFERENCE.items()}) }")

In [ ]:
# Feature extraction, resumable: every chunk of 100 images is saved, and a rerun skips the chunks already there.
CHUNK = 100
chunk_dir = WORK / f"features_{IMAGE_SIZE}px_{'context' if USE_CONTEXT else 'no_context'}"
chunk_dir.mkdir(parents=True, exist_ok=True)
seconds_per_pass, failed = [], 0
for start in range(0, len(pilot), CHUNK):
    path = chunk_dir / f"chunk_{start:06d}.npz"
    if path.exists():
        continue
    arrays = {f"{side}_{depth}_{pool}": [] for side in (0, 1) for depth in PROBE for pool in POOLS}
    readable = []
    for _, row in pilot.iloc[start : start + CHUNK].iterrows():
        try:
            base = prepared(row["image_file"])
            text = prompt_text(row)
            sides = []
            for image in (base, ImageOps.mirror(base)):  # the mirror is pixel-exact: same JPEG, flipped
                began = time.perf_counter()
                sides.append(read_once(image, text)[0])
                seconds_per_pass.append(time.perf_counter() - began)
        except Exception as error:  # one broken file must not stop the run; many must
            failed += 1
            print(f"skipped an image: {type(error).__name__}")
            if failed > 25 and failed > 0.05 * (start + CHUNK):
                raise
            sides = [{depth: {pool: np.zeros_like(REFERENCE[pool]) for pool in POOLS} for depth in PROBE}] * 2
            readable.append(False)
        else:
            readable.append(True)
        for side, features in enumerate(sides):
            for depth in PROBE:
                for pool in POOLS:
                    arrays[f"{side}_{depth}_{pool}"].append(features[depth][pool])
    np.savez(path, readable=np.array(readable), **{key: np.stack(value) for key, value in arrays.items()})
    done = min(start + CHUNK, len(pilot))
    recent = float(np.mean(seconds_per_pass[-2 * CHUNK :])) if seconds_per_pass else float("nan")
    minutes_left = (len(pilot) - done) * 2 * recent / 60
    print(f"{done}/{len(pilot)} images | {recent:.2f} s per pass | about {minutes_left:.0f} min left")

In [ ]:
chunks = [np.load(path) for path in sorted(chunk_dir.glob("chunk_*.npz"))]
READABLE = np.concatenate([chunk["readable"] for chunk in chunks])
if len(READABLE) != len(pilot):
    raise RuntimeError(f"{len(READABLE)} feature rows for {len(pilot)} images: rerun the extraction cell")
F = {
    (side, depth, pool): np.concatenate([chunk[f"{side}_{depth}_{pool}"] for chunk in chunks]).astype(np.float32)
    for side in (0, 1)
    for depth in PROBE
    for pool in POOLS
}
SECONDS_PER_PASS = float(np.median(seconds_per_pass)) if seconds_per_pass else float("nan")
PEAK_GB = [round(torch.cuda.max_memory_allocated(index) / 2**30, 1) for index in range(torch.cuda.device_count())]
print(f"{int(READABLE.sum())} readable images, {SECONDS_PER_PASS:.2f} s per pass (median), peak GPU GB {PEAK_GB}")

## The heads

Every feature block (a depth and a pool) is reduced by PCA, then one logistic regression per dimension is fitted
with class weights tempered for macro F1 (power 0.5). Every score below is **out-of-fold**: each image is predicted
by a head that never saw its product. The best depth and pool per dimension are picked on these same folds, so the
best numbers are a little optimistic; the official holdout is untouched for the real EXP rows.

Perspective gets three variants, because left and right is the one thing a mirror changes:

- **plain**: the image's own features.
- **mirror**: also trains on the mirrored copies with left and right swapped, and averages each test image with its
  mirror (the mirror's answer swapped back).
- **split**: the features of the image plus its mirror (what a mirror leaves alone: front, back, top) next to their
  difference (what a mirror flips: left and right), trained and averaged like "mirror".

In [ ]:
from joblib import Parallel, delayed
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler

PRODUCTS = pilot["product_id"].to_numpy()
_order = pilot["product_id"].drop_duplicates().sample(frac=1.0, random_state=SEED).tolist()
FOLD = pilot["product_id"].map({product: index % FOLDS for index, product in enumerate(_order)}).to_numpy()
ROWS = {dimension: np.flatnonzero(scored(pilot, dimension) & READABLE) for dimension in DIMENSIONS}
LABELS = {dimension: pilot[dimension].to_numpy(dtype=object)[ROWS[dimension]] for dimension in DIMENSIONS}


def reduce(*blocks, dims=PCA_DIMS):
    """PCA fitted on the blocks stacked (no labels involved), applied to each block."""
    stacked = np.vstack(blocks)
    components = min(dims, stacked.shape[0] - 1, stacked.shape[1])
    pca = PCA(n_components=components, svd_solver="randomized", random_state=SEED).fit(stacked)
    return [pca.transform(block).astype(np.float32) for block in blocks]


def tempered_weights(y):
    classes, counts = np.unique(y, return_counts=True)
    return dict(zip(classes, (len(y) / (len(classes) * counts)) ** 0.5))


def fit_predict(x_train, y_train, tests, c):
    """Class probabilities for every matrix in tests, and the classes in column order."""
    classes = np.unique(y_train)
    if len(classes) == 1:
        return [np.ones((len(test), 1)) for test in tests], classes
    scaler = StandardScaler().fit(x_train)
    head = LogisticRegression(C=c, max_iter=3000, class_weight=tempered_weights(y_train))
    head.fit(scaler.transform(x_train), y_train)
    return [head.predict_proba(scaler.transform(test)) for test in tests], head.classes_


def out_of_fold(x, y, fold, products, c, mirror=None, fraction=1.0):
    """The out-of-fold class and its probability for every row.

    mirror: the features of the mirrored images, row for row. The head then also trains on them with left and right
    swapped, and each test image is averaged with its mirror. fraction: the share of training products kept, for the
    learning curve.
    """
    predicted, confidence = np.empty(len(y), dtype=object), np.zeros(len(y))
    for k in range(FOLDS):
        test = fold == k
        if not test.any():
            continue
        train_rows = np.flatnonzero(~test)
        if fraction < 1.0:
            kept = np.unique(products[train_rows])
            kept = np.random.default_rng(SEED + k).choice(kept, max(1, round(fraction * len(kept))), replace=False)
            train_rows = train_rows[np.isin(products[train_rows], kept)]
        x_train, y_train, tests = x[train_rows], y[train_rows], [x[test]]
        if mirror is not None:
            x_train = np.vstack([x_train, mirror[train_rows]])
            y_train = np.concatenate([y_train, np.array([swap_side(label) for label in y_train], dtype=object)])
            tests.append(mirror[test])
        probabilities, classes = fit_predict(x_train, y_train, tests, c)
        p = probabilities[0]
        if mirror is not None:
            column = {label: index for index, label in enumerate(classes)}
            p = 0.5 * (p + probabilities[1][:, [column.get(swap_side(label), i) for i, label in enumerate(classes)]])
        predicted[test], confidence[test] = classes[p.argmax(1)], p.max(1)
    return predicted, confidence


def macro_f1(y, predicted):
    """metric.py's per-dimension score: the classes are the labels plus every class predicted."""
    return float(f1_score(list(y), list(predicted), average="macro", zero_division=0))

In [ ]:
REDUCED = {}
for depth in PROBE:
    for pool in POOLS:
        REDUCED[(0, depth, pool)], REDUCED[(1, depth, pool)] = reduce(F[(0, depth, pool)], F[(1, depth, pool)])


def score_config(dimension, depth, pool, c):
    rows = ROWS[dimension]
    predicted, _ = out_of_fold(REDUCED[(0, depth, pool)][rows], LABELS[dimension], FOLD[rows], PRODUCTS[rows], c)
    return {
        "dimension": dimension,
        "depth": depth,
        "pool": pool,
        "C": c,
        "macro_f1": macro_f1(LABELS[dimension], predicted),
    }


jobs = [(dimension, depth, pool, c) for dimension in DIMENSIONS for depth in PROBE for pool in POOLS for c in C_GRID]
grid = pd.DataFrame(Parallel(n_jobs=-1, prefer="threads")(delayed(score_config)(*job) for job in jobs))
best = grid.loc[grid.groupby("dimension")["macro_f1"].idxmax()].set_index("dimension").loc[DIMENSIONS]
by_depth = grid.groupby(["dimension", "depth"])["macro_f1"].max().unstack("depth").loc[DIMENSIONS]
print(best)
print(by_depth.round(3))

In [ ]:
variants = []
if PERSPECTIVE is not None:
    rows, y = ROWS[PERSPECTIVE], LABELS[PERSPECTIVE]
    top = grid[grid["dimension"] == PERSPECTIVE].sort_values("macro_f1", ascending=False)
    for _, config in top.drop_duplicates(["depth", "pool"]).head(3).iterrows():
        depth, pool = int(config["depth"]), config["pool"]
        own, mirrored = REDUCED[(0, depth, pool)][rows], REDUCED[(1, depth, pool)][rows]
        (same,) = reduce(F[(0, depth, pool)] + F[(1, depth, pool)], dims=PCA_DIMS // 2)
        # PCA fitted on the difference and its negative has mean zero, so it maps -difference to -its components:
        # the mirrored row is exactly (same, -flipped).
        flipped, flipped_back = reduce(
            F[(0, depth, pool)] - F[(1, depth, pool)], F[(1, depth, pool)] - F[(0, depth, pool)], dims=PCA_DIMS // 2
        )
        split, split_mirror = np.hstack([same, flipped])[rows], np.hstack([same, flipped_back])[rows]
        for c in C_GRID:
            for name, x, mirror in (("plain", own, None), ("mirror", own, mirrored), ("split", split, split_mirror)):
                predicted, confidence = out_of_fold(x, y, FOLD[rows], PRODUCTS[rows], c, mirror=mirror)
                variants.append(
                    {
                        "variant": name,
                        "depth": depth,
                        "pool": pool,
                        "C": c,
                        "macro_f1": macro_f1(y, predicted),
                        "predicted": predicted,
                        "confidence": confidence,
                        "x": x,
                        "mirror": mirror,
                    }
                )
    shown = pd.DataFrame(variants)[["variant", "depth", "pool", "C", "macro_f1"]]
    print(shown.sort_values("macro_f1", ascending=False).head(9).round(3))

In [ ]:
# The final pick per dimension, its out-of-fold answers, and the learning curve: the same heads trained on a quarter,
# a half and all of the pilot's training products. A steep curve means the full train.csv (about 8x more) will help.
PICK, ANSWERS = {}, {}
for dimension in DIMENSIONS:
    rows, y = ROWS[dimension], LABELS[dimension]
    choice = best.loc[dimension]
    PICK[dimension] = {
        "variant": "plain",
        "depth": int(choice["depth"]),
        "pool": choice["pool"],
        "C": float(choice["C"]),
    }
    x, mirror = REDUCED[(0, PICK[dimension]["depth"], PICK[dimension]["pool"])][rows], None
    if dimension == PERSPECTIVE and variants:
        winner = max(variants, key=lambda row: row["macro_f1"])
        PICK[dimension] = {key: winner[key] for key in ("variant", "depth", "pool", "C")}
        x, mirror = winner["x"], winner["mirror"]
    curve = {}
    for fraction in (0.25, 0.5, 1.0):
        predicted, confidence = out_of_fold(x, y, FOLD[rows], PRODUCTS[rows], PICK[dimension]["C"], mirror, fraction)
        curve[fraction] = macro_f1(y, predicted)
    PICK[dimension].update(macro_f1=curve[1.0], curve=curve, scored=len(rows), classes_seen=len(set(y)))
    ANSWERS[dimension] = (predicted, confidence)
    print(dimension, PICK[dimension])

In [ ]:
# Cross-check against the competition's own metric.py on the pilot, and count likely label errors (out-of-fold
# answers that disagree with the label at 90% or more), for the bonus list later. Counts only, no ids.
solution, submission = pilot[["image_id"]].copy(), pilot[["image_id"]].copy()
LIKELY_ERRORS = {}
for dimension in DIMENSIONS:
    rows, y = ROWS[dimension], LABELS[dimension]
    predicted, confidence = ANSWERS[dimension]
    answer = np.full(len(pilot), pd.Series(y).mode().iloc[0], dtype=object)  # unscored cells do not count
    answer[rows] = predicted
    submission[dimension] = answer
    truth = np.full(len(pilot), None, dtype=object)
    truth[rows] = y
    solution[dimension] = truth
    LIKELY_ERRORS[dimension] = int(((predicted != y) & (confidence >= 0.9)).sum())
MEAN_F1 = float(np.mean([PICK[dimension]["macro_f1"] for dimension in DIMENSIONS]))
try:
    METRIC_SCORE = float(metric.score(solution, submission, "image_id"))
except Exception as error:  # noqa: BLE001 - the cross-check must never hide the result above it
    METRIC_SCORE = float("nan")
    print(f"metric.py refused the pilot frame: {type(error).__name__}: {error}")
print(f"mean macro F1 {MEAN_F1:.4f} | metric.py {METRIC_SCORE:.4f} | likely label errors {LIKELY_ERRORS}")

In [ ]:
import json

if MEAN_F1 >= 0.50:
    VERDICT = "GO: scale to all of train.csv and the official holdout"
elif MEAN_F1 >= 0.35:
    VERDICT = "LOOK: read the learning curve and the weakest dimensions before scaling"
else:
    VERDICT = "STOP: frozen features are not enough on their own"
early_exit = max(PICK[dimension]["depth"] for dimension in DIMENSIONS)
lines = [
    "### One-look pilot, Kaggle run",
    "",
    (
        f"- **Mean macro F1 (out-of-fold, pilot): {MEAN_F1:.3f}**, metric.py on the same answers {METRIC_SCORE:.3f}. "
        f"Verdict: **{VERDICT}**."
    ),
    (
        f"- Model `{MODEL_ID}` at `{MODEL_REVISION[:12]}`, transformers {transformers.__version__}, {DTYPE}, "
        f"{len(GPUS)} x {GPUS[0]}."
    ),
    (
        f"- Pilot: {len(pilot)} images of {pilot['product_id'].nunique()} products outside the official holdout, "
        f"{FOLDS} folds by product, {int((~READABLE).sum())} unreadable, context {'on' if USE_CONTEXT else 'off'}."
    ),
    (
        f"- Speed: {SECONDS_PER_PASS:.2f} s per pass at about {TOKENS} tokens, all {N_LAYERS} layers "
        f"({1 / SECONDS_PER_PASS:.2f} images/s without the mirror, half that with it). The deepest depth any pick "
        f"uses is {early_exit}, so an early exit there skips {N_LAYERS - early_exit} layers. Peak GPU GB {PEAK_GB}."
    ),
    "",
    "| Dimension | Pick | Depth | Pool | Macro F1 | at 25% / 50% of the data | Scored | Classes seen | Label errors? |",
    "| --- | --- | --- | --- | --- | --- | --- | --- | --- |",
]
for dimension in DIMENSIONS:
    pick = PICK[dimension]
    lines.append(
        f"| {dimension} | {pick['variant']} | {pick['depth']} | {pick['pool']} | {pick['macro_f1']:.3f} | "
        f"{pick['curve'][0.25]:.3f} / {pick['curve'][0.5]:.3f} | {pick['scored']} | "
        f"{pick['classes_seen']}/{len(CLASSES[dimension])} | {LIKELY_ERRORS[dimension]} |"
    )
lines += ["", "Best macro F1 by depth (any pool):", "", "| Dimension | " + " | ".join(str(d) for d in PROBE) + " |"]
lines.append("| --- |" + " --- |" * len(PROBE))
for dimension in DIMENSIONS:
    lines.append(f"| {dimension} | " + " | ".join(f"{by_depth.loc[dimension, d]:.3f}" for d in PROBE) + " |")
if variants:
    lines += [
        "",
        "Perspective, best C per variant and depth:",
        "",
        "| Variant | Depth | Pool | Macro F1 |",
        "| --- | --- | --- | --- |",
    ]
    table = (
        pd.DataFrame(variants).sort_values("macro_f1", ascending=False).drop_duplicates(["variant", "depth", "pool"])
    )
    for _, row in table.iterrows():
        lines.append(f"| {row['variant']} | {row['depth']} | {row['pool']} | {row['macro_f1']:.3f} |")
report = "\n".join(lines)
(WORK / "pilot_report.md").write_text(report, encoding="utf-8")
(WORK / "pilot_results.json").write_text(
    json.dumps(
        {"mean_macro_f1": MEAN_F1, "metric_score": METRIC_SCORE, "pick": PICK, "verdict": VERDICT},
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)
print("=" * 30, "PASTE EVERYTHING BELOW INTO THE PR", "=" * 30)
print(report)